# 실습 1. Python 텍스트 데이터 처리 (Day 1 / 90분)

> 시나리오: **쇼핑몰 리뷰 데이터 정제** — `data/reviews.csv`
>
> Day 1(Pandas·정규식)은 이 저장소의 **로컬 노트북**으로 진행한다.

## Task
1. CSV 로드 후 **결측·이상치 리포트** 작성
2. `text` 컬럼 정제 — URL/특수문자/공백 정규화
3. **정규식**으로 광고 패턴(`[광고]`, `^협찬`, `구매처:`) 제거
4. `rating` 별 평균 텍스트 길이 / 단어 수 집계
5. 정제 결과를 `reviews_clean.parquet` 으로 저장

In [8]:
import re
# 이메일 추출
text = "unzena@naver.com 010-86711121 5669fab@naver.com"
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
print(emails)

# 전화번호 정규화 (010-1234-5678 / 01012345678 / 010 1234 5678)
phone = re.sub(r"[\s-]", "", "010 1234-5678") # "01012345678"

print(phone)
# 한글 추출
korean = re.findall(r"[가-힣]+", "Hello 안녕 World 세상")
print(korean)
# ['안녕', '세상']
# 명명된 그룹 — 가독성 ↑
m = re.match(r"(?P<year>\d{4})-(?P<month>\d{2})", "2026-05")
m.group("year") # "2026"
print(m.group("year")) # "2026"
print(m.group("month")) # "05"

['unzena@naver.com', '5669fab@naver.com']
01012345678
['안녕', '세상']
2026
05


In [10]:
import pandas as pd

# data/reviews.csv 가 없으면: 터미널에서 `python data/make_reviews.py`
df = pd.read_csv("../data/reviews.csv")
print(df.shape)
df.head()

(10000, 5)


,id,user,text,rating,created_at
0,1,user087,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 ????? 👍,2.0,2026/06/18 02:37
1,2,user072,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,4.0,23.02.2026
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14T10:17:00
3,4,user045,[광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡,4.0,NaN
4,5,user047,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,5.0,2026-05-07


## 1) 결측·이상치 리포트

In [48]:
df.info()
print("\n결측치:\n", df.isna().sum())
print("\nrating 분포:\n", df["rating"].value_counts(dropna=False))

# created_at 은 형식이 제각각 → errors='coerce' 로 파싱 실패는 NaT 처리
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", format="mixed")
print("\n날짜 파싱 실패(NaT):", df["created_at"].isna().sum(), "건")

# NaT 행 제거
print("제거 전 행 수:", len(df))
df = df[df["created_at"].notna()]
print("제거 후 행 수:", len(df))


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          10000 non-null  int64         
 1   user        10000 non-null  str           
 2   text        10000 non-null  str           
 3   rating      9494 non-null   float64       
 4   created_at  8000 non-null   datetime64[us]
 5   clean_text  10000 non-null  str           
 6   length      10000 non-null  int64         
 7   words       10000 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 2.1 MB

결측치:
 id               0
user             0
text             0
rating         506
created_at    2000
clean_text       0
length           0
words            0
dtype: int64

rating 분포:
 rating
4.0    2387
5.0    2363
1.0    1689
2.0    1663
3.0    1392
NaN     506
Name: count, dtype: int64

날짜 파싱 실패(NaT): 2000 건
제거 전 행 수: 10000
제거 후 행 수: 8000


## 2) 텍스트 정제 파이프라인 (슬라이드 `clean()` 참고)

`.str` 로 컬럼 전체를 한 번에 정제한다. 정규식 패턴 풀이:
- `https?://\S+` → http/https 로 시작하는 URL (공백 전까지)
- `[^\w가-힣\s]` → 단어문자(`\w`)·한글(`가-힣`)·공백(`\s`) **이 아닌** 것 = 특수문자/이모지
- `\s+` → 연속 공백을 1칸으로

In [61]:
import re

def clean(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .str.strip()
         .str.replace(r"https?://\S+", "", regex=True)        # URL 제거
         .str.replace(r"[^\w가-힣\s]", " ", regex=True)        # 특수문자 제거 (가-힣 = U+AC00~U+D7A3)
         .str.replace(r"\s+", " ", regex=True)                 # 공백 정규화
         .str.strip()
    )

# 먼저 원본 text에서 광고 패턴 제거
AD = re.compile(r"\[광고\]|^협찬|구매처\s*:")
print(f"광고성 리뷰: {df['text'].str.contains(AD).sum()}건")
df["text_no_ad"] = df["text"].str.replace(AD, "", regex=True)

# 그 후 정제 함수 적용
df["clean_text"] = clean(df["text_no_ad"])
df[["text", "text_no_ad", "clean_text"]].head()


광고성 리뷰: 3942건


,text,text_no_ad,clean_text
0,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 ????? 👍,받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 ????? 👍,받아 작성한 후기입니다 생각보다 너무 작고 부실해요
1,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요
2,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로
4,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요
5,[광고] 지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍,지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍,지금 구매하면 사은품 증정 가성비 최고입니다 또 살게요


## 3) 광고 패턴 제거 (정규식)

In [62]:
AD = re.compile(r"\[광고\]|^협찬|구매처\s*:")
is_ad = df["text"].str.contains(AD)

# 광고 제거 전후 비교
print("\n광고 문구 제거 전후 비교:")
ad_samples = df[is_ad][["text", "text_no_ad", "clean_text"]].head()
print(ad_samples)



광고 문구 제거 전후 비교:
                                                 text  \
0            협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 ????? 👍   
1               협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍   
4          [광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ   
5              [광고] 지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍   
14  구매처: http://shop.example.com 방문하세요 배송이 정말 빨라서 ...   

                                           text_no_ad  \
0               받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 ????? 👍   
1                  받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍   
4               지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ   
5                   지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍   
14   http://shop.example.com 방문하세요 배송이 정말 빨라서 깜짝 놀...   

                           clean_text  
0        받아 작성한 후기입니다 생각보다 너무 작고 부실해요  
1     받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요  
4   지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요  
5      지금 구매하면 사은품 증정 가성비 최고입니다 또 살게요  
14           방문하세요 배송이 정말 빨라서 깜짝 놀랐어요  


## 4) rating 별 텍스트 길이 / 단어 수 집계

In [63]:
df["length"] = df["clean_text"].str.len()
df["words"] = df["clean_text"].str.split().str.len()
df.groupby("rating")[["length", "words"]].mean()

,length,words
rating,,
1.0,23.829587,6.150834
2.0,23.601657,6.099398
3.0,19.806338,5.278169
4.0,23.520231,5.827115
5.0,23.374054,5.805946


## 5) 저장 — 실습 2의 입력이 된다

In [64]:
df.to_parquet("../data/reviews_clean.parquet", index=False)
print("saved: data/reviews_clean.parquet")

saved: data/reviews_clean.parquet
